# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/capstone/starter_D_pubmed_rct.ipynb)

# BME i9400 — Capstone starter D
## The Role of a Sentence in a Medical Abstract

This notebook downloads the data, applies the fixed split, and fits the baseline. Run it from top to
bottom before you change anything; it takes about two minutes.

---
## 1 — The data

PubMed 20k RCT (Dernoncourt and Lee, "PubMed 200k RCT: a dataset for sequential sentence
classification in medical abstracts", *IJCNLP*, 2017): 20,000 abstracts of randomised controlled
trials from PubMed, in which every sentence is labelled with its role in the abstract.

- **A row of $X$**: one sentence, with the identifier of its abstract, its position in the abstract
  (counting from 0), and the number of sentences in the abstract.
- **$y$**: the role of the sentence: `BACKGROUND`, `OBJECTIVE`, `METHODS`, `RESULTS`, or `CONCLUSIONS`.

> The labels come from the section headings that the authors themselves used in structured
> abstracts; the headings were then removed from the text.

In [ ]:
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

BASE = "https://raw.githubusercontent.com/Franck-Dernoncourt/pubmed-rct/master/PubMed_20k_RCT/{}.txt"

def load(split):
    rows, abstract, buffer = [], None, []
    for line in urllib.request.urlopen(BASE.format(split)).read().decode().splitlines():
        if line.startswith("###"):
            abstract, buffer = line[3:], []
        elif line == "":
            rows += [(abstract, i, len(buffer), lab, s) for i, (lab, s) in enumerate(buffer)]
            buffer = []
        else:
            buffer.append(tuple(line.split("\t", 1)))
    return pd.DataFrame(rows, columns=["abstract_id", "position", "n_sentences", "label", "sentence"])

train, dev, test = load("train"), load("dev"), load("test")
for name, d in [("training", train), ("development", dev), ("test", test)]:
    print(f"{name:12s} {d['abstract_id'].nunique():6d} abstracts  {len(d):7d} sentences")
print(train["label"].value_counts(normalize=True).round(3).to_string())
train.head(8)

---
## 2 — The fixed split

The official split is used unchanged: training, development (the validation set), and test, divided
by abstract, so that no abstract contributes sentences to two sets.

> Sentences of the same abstract are not independent: they share a topic, a vocabulary, and an
> author. For this reason the bootstrap of section 4 resamples abstracts, not sentences.

---
## 3 — The baseline

TF-IDF features of single words and pairs of consecutive words (bigrams) that occur in at least two
training sentences, and multinomial logistic regression ($C = 1$). The sentence is used alone: neither
its position nor its neighbours.

> **TF-IDF** (term frequency – inverse document frequency) represents a sentence as a vector with one
> entry per word or bigram in the vocabulary, large when the term occurs in the sentence but is rare
> across the training sentences.
>
> The **primary metric** is the macro-averaged F1 score over the five classes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
A_train = tfidf.fit_transform(train["sentence"])
A_dev   = tfidf.transform(dev["sentence"])

baseline = LogisticRegression(C=1.0, max_iter=2000).fit(A_train, train["label"])
pred_dev_base = baseline.predict(A_dev)
print(f"{A_train.shape[1]} TF-IDF features")
print(f"baseline, development set: macro F1 {f1_score(dev['label'], pred_dev_base, average='macro'):.3f}")
print(classification_report(dev["label"], pred_dev_base, digits=3))
labels = sorted(train["label"].unique())
print(pd.DataFrame(confusion_matrix(dev["label"], pred_dev_base, labels=labels), index=labels, columns=labels))

---
## 4 — The test set and the bootstrap

> **The test set is used once.** The cell below defines the test-set evaluation and the bootstrap
> confidence interval but does not run them. Run them at the end of the project, for your final model
> and for the baseline only, after every other decision has been made on validation data.

In [ ]:
def bootstrap_difference(metric, y, pred_new, pred_base, groups=None, n_boot=2000, seed=0):
    """95% bootstrap confidence interval for metric(y, pred_new) - metric(y, pred_base).

    Resamples whole groups (patients, records, abstracts) with replacement when `groups` is given,
    otherwise single examples. Both models are scored on the same resample, so the interval is for
    the paired difference (Meeting 20).
    """
    rng = np.random.default_rng(seed)
    y, pred_new, pred_base = np.asarray(y), np.asarray(pred_new), np.asarray(pred_base)
    if groups is None:
        groups = np.arange(len(y))
    uniq, inv = np.unique(np.asarray(groups), return_inverse=True)
    members = [np.flatnonzero(inv == k) for k in range(len(uniq))]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        diffs.append(metric(y[idx], pred_new[idx]) - metric(y[idx], pred_base[idx]))
    point = metric(y, pred_new) - metric(y, pred_base)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return point, lo, hi


def evaluate_on_test(pred_test_new, pred_test_base):
    """Macro F1 on the test set for both models, and the 95% CI for the difference, resampling abstracts."""
    # labels as integer codes: scoring 2,000 resamples of 30,000 sentences is several times faster than with strings
    code = {lab: k for k, lab in enumerate(labels)}
    yt, pn, pb = (np.array([code[v] for v in arr]) for arr in (test["label"], pred_test_new, pred_test_base))
    macro_f1 = lambda a, b: f1_score(a, b, average="macro", labels=range(len(labels)))
    for name, p in [("final model", pn), ("baseline", pb)]:
        print(f"{name:12s} macro F1 {macro_f1(yt, p):.3f}")
    d, lo, hi = bootstrap_difference(macro_f1, yt, pn, pb, groups=test["abstract_id"])
    print(f"difference in macro F1 {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")

# At the end of the project, and only then:
# evaluate_on_test(final_model_predictions_on_test, baseline.predict(tfidf.transform(test["sentence"])))

### Computing time

Encoding the 180,040 training sentences with a small pretrained encoder such as
`sentence-transformers/all-MiniLM-L6-v2` takes on the order of 15–30 minutes on a free Colab CPU.
Develop on a random subsample of training **abstracts** (for example 3,000 abstracts, about 36,000
sentences), save the embeddings to Google Drive with `np.save` so that they are computed once, and
report the size of the subsample used for the final model.

---
## 5 — From here

The requirements, deliverables, and grading are in the capstone brief,
[`capstone/README.md`](https://github.com/dmochow/BME-i9400-2026/blob/main/capstone/README.md). In
short: reproduce the baseline above; search at least two further model families with every choice
made on validation data; evaluate the final model and the baseline once on the test set, with a 95%
bootstrap confidence interval for the difference; analyse the errors; write the model card.

Make a copy of this notebook (`File ▸ Save a copy in Drive`) and build your study in it.

Directions specific to this dataset: the position of the sentence in its abstract; features of the
previous and next sentences; frozen sentence embeddings from a pretrained encoder (Meeting 24) with a
classifier on top; and the confusion between `BACKGROUND` and `OBJECTIVE`.